# NFTs: The Token, the Owner, and the Picture

NFTs are often shown as cartoon monkeys or animated GIFs. That can make them look like pictures with unusually high price tags. But what does buying one actually give you?

**NFT stands for non-fungible token.** It is a digital token with its own ID and a recorded owner. It can be linked to a picture or another item. The blockchain tracks ownership of the token; the picture can still be copied.

For a fictional example, Alice takes a class graduation photo and creates a token linked to it:

- Token ID: 1
- Owner: Alice
- Picture: a link to the graduation photo

Creating a token is called **minting**. If Alice transfers this token to Bob, its owner changes to Bob. If Jacob saves the photo, he gets a copy of the photo. The token's owner does not change.

A **collection** can contain many NFTs. Several can even link to the same picture. Each still has its own token ID, owner, and transfer history. We will show this in the code.

We will also look at purchases, creator royalties, and what happens when the website storing the picture disappears. The explanations work on their own; the Python examples let you try each step.


## Fungible means interchangeable

For an ordinary payment, bitcoin is treated as **fungible**: one amount can replace the same amount. If I lend you 0.01 BTC, I expect 0.01 BTC back. I do not need the exact units I lent you.

If I lend you my phone, I want that particular phone back. Another phone of the same model has different photos and scratches. Also, I have spent years arranging those apps badly. This is what **non-fungible** means: another item does not automatically replace the particular one.

| Holding | What we need to know |
| --- | --- |
| Alice has 0.01 BTC | How much bitcoin she has |
| Alice owns NFT #1 in a particular collection | Which token she owns |

A **smart contract** is a program that applies rules to the collection, such as who can mint a token or transfer it. In this ERC-721-style model, each token ID has one current owner.

There is one token #1 in this collection. Minting #2 creates another individually tracked token. The collection could have 10,000 tokens, including many linked to the same photo. They can look the same and sell for the same price while still having separate IDs and owners.

**Scarcity and non-fungibility answer different questions.** Scarcity concerns how many exist. Non-fungibility concerns whether one item can replace another. A unique ID alone does not promise that nobody will mint more tokens linked to the picture.


## What information does the token hold?

Our collection records an owner for each token. It also stores **metadata**: information describing what the token refers to. Here that means a title, a description, a link to the file, and a file hash.

A **URI** is a location string, such as a web address. A **hash** is a fingerprint calculated from the file's bytes. We use SHA-256, which gives us 64 hexadecimal characters. We can hash a downloaded file and compare the result with the stored fingerprint to check whether the bytes match.

NFTs can store their content in different ways. Many use ERC-721, a common NFT interface standard, with a `tokenURI` pointing to a metadata document that links to the image. Some store art on-chain. Our separate SHA-256 field is included to make file verification easy to demonstrate. [ERC-721 specification](https://eips.ethereum.org/EIPS/eip-721).

For the code, we import these helpers:

- `Blockchain` keeps an ordered log of blocks containing transaction data.
- `Transaction` represents a request, and `Network` passes requests between nodes.
- `Validator` represents a participant allowed to propose a block.
- `SmartContract` gives a program an address and lets us call a method by name.
- `submit_call` records a requested call in a block, then runs that method.

We define the NFT-specific classes below. `TokenMetadata` holds the descriptive fields. The two error classes explain when a token does not exist or the caller is not its owner.


In [1]:
import hashlib
import random
import statistics
from dataclasses import dataclass
from decimal import Decimal

from blockchain_lib.contracts import SmartContract, submit_call
from blockchain_lib.mempool import Network, Transaction
from blockchain_lib.pos import Blockchain, Validator

# Shared helpers supply the chain, transaction queue, and method dispatch.
# The NFT-specific classes and ownership rules are defined below.

In [2]:
@dataclass(frozen=True)
class TokenMetadata:
    """Describe the off-chain artifact without storing its file bytes.

    Attributes:
        name: Human-readable title.
        description: What the creator says the artifact depicts.
        content_hash: SHA-256 digest of the artifact bytes in this model.
        uri: Location used to retrieve the artifact in this model.
    """

    name: str
    description: str
    content_hash: str
    uri: str


class NotTheOwnerError(PermissionError):
    """Raised when someone tries to move another holder's token."""


class TokenDoesNotExistError(LookupError):
    """Raised when a token ID has not been minted in this collection."""

## The rules for minting and transferring

`NFTCollection` keeps an owner and metadata for each token ID. Its methods do five jobs:

1. `mint` creates a new token and records its owner and metadata. Our collection allows only its creator to mint.
2. `owner_of` and `metadata_of` read a token's record.
3. `transfer` checks the current owner, then updates that token's owner.
4. `royalty_info` and `sell` calculate a creator's share of a proposed sale.
5. `verify_content` checks file bytes against the stored hash.

The royalty rate uses **basis points**: `500 / 10,000 = 5%`. `Decimal("2")` keeps our decimal arithmetic exact. The sale examples use ETH, or ether, Ethereum's native currency. The prices are made up.

This Python class demonstrates the ownership idea behind ERC-721. A complete implementation also includes approvals, operators, and safe-transfer checks. We omit those here. Our extra `sell` method calculates a proposed payout split; it does not send money. [ERC-721](https://eips.ethereum.org/EIPS/eip-721), [ERC-2981](https://eips.ethereum.org/EIPS/eip-2981).


In [3]:
class NFTCollection(SmartContract):
    """Track each token separately, with a creator and royalty quotation.

    Caller names stand in for authenticated wallet addresses. Ownership
    lives in Python memory; blocks record requested calls.
    """

    def __init__(self, address: str, creator: str, royalty_bps: int = 500) -> None:
        """Set the creator and rate; 500 basis points means five percent."""
        super().__init__(address)
        if not creator:
            raise ValueError("A collection needs a creator.")
        if not 0 <= royalty_bps <= 10_000:
            raise ValueError("Royalty basis points must be between 0 and 10,000.")
        self.creator = creator
        self.royalty_bps = royalty_bps
        self._next_id = 1
        self._owners: dict[int, str] = {}
        self._metadata: dict[int, TokenMetadata] = {}

    def mint(self, to: str, metadata: TokenMetadata, caller: str) -> int:
        """Create a token; this collection lets only its creator mint."""
        if caller != self.creator:
            raise PermissionError(f"Only {self.creator} can mint in this collection.")
        if not to:
            raise ValueError("A token needs a recipient.")
        token_id = self._next_id
        self._next_id += 1
        self._owners[token_id] = to
        self._metadata[token_id] = metadata
        return token_id

    def owner_of(self, token_id: int) -> str:
        """Return this token's owner, or reject an unminted ID."""
        if token_id not in self._owners:
            raise TokenDoesNotExistError(f"Token #{token_id} does not exist.")
        return self._owners[token_id]

    def metadata_of(self, token_id: int) -> TokenMetadata:
        """Read the pointer and hash attached to an existing token."""
        self.owner_of(token_id)
        return self._metadata[token_id]

    def transfer(self, token_id: int, caller: str, to: str) -> None:
        """Move a token only when its current owner is the caller."""
        current = self.owner_of(token_id)
        if caller != current:
            raise NotTheOwnerError(
                f"{caller} does not own token #{token_id}; {current} does."
            )
        if not to:
            raise ValueError("A transfer needs a recipient.")
        self._owners[token_id] = to

    def royalty_info(
        self, token_id: int, sale_price_eth: Decimal
    ) -> tuple[str, Decimal]:
        """Quote a royalty recipient and amount; no money moves here."""
        self.owner_of(token_id)
        if not sale_price_eth.is_finite() or sale_price_eth < 0:
            raise ValueError("Sale price must be finite and non-negative.")
        amount = sale_price_eth * Decimal(self.royalty_bps) / Decimal(10_000)
        return self.creator, amount

    def sell(
        self, token_id: int, caller: str, to: str, price_eth: Decimal
    ) -> dict[str, str | int | Decimal]:
        """Transfer a token and return its proposed payout split.

        This classroom helper computes allocations, not ETH settlement.
        It is separate from the basic transfer route, which has no price.
        """
        creator, royalty = self.royalty_info(token_id, price_eth)
        if price_eth == 0:
            raise ValueError("Use transfer for a gift; sale price must be positive.")
        # Validate the price before any ownership changes.
        self.transfer(token_id, caller, to)
        return {
            "token_id": token_id,
            "seller": caller,
            "buyer": to,
            "price_eth": price_eth,
            "seller_receives": price_eth - royalty,
            "creator": creator,
            "creator_royalty": royalty,
        }

    def verify_content(self, token_id: int, file_bytes: bytes) -> bool:
        """Check byte integrity, without deciding authorship or availability."""
        metadata = self.metadata_of(token_id)
        return hashlib.sha256(file_bytes).hexdigest() == metadata.content_hash

## Prepare the graduation photo example

We will use a class graduation photo as the example. To run the code without downloading an image, we use a small text caption as the file. A JPEG would follow the same steps: read its bytes, calculate a hash, and store a link to the file.

The file stays **off-chain**, meaning outside the blockchain. The `off_chain_files` dictionary represents our hosting service. The `.invalid` address is fictional; the code makes no network requests.

We also create a small chain and record the collection's creation in a block. The contract address, `class-photos.nft`, is a readable label for this model.

> **Before running:** Does calculating a hash store the file itself? Could the fingerprint recreate the photo?


In [4]:
artifact_name = "class_graduation.txt"
artifact_bytes = (
    b"Class graduation photo.\n"
    b"Students in gowns, certificates in hand.\n"
    b"One group photo. Someone always blinks.\n"
)
artifact_hash = hashlib.sha256(artifact_bytes).hexdigest()
artifact_uri = "https://gallery.invalid/class-graduation.txt"
off_chain_files = {artifact_uri: artifact_bytes}

metadata = TokenMetadata(
    name="Class Graduation Photo",
    description="The whole class together. Almost everyone has their eyes open.",
    content_hash=artifact_hash,
    uri=artifact_uri,
)

network = Network(["Node A", "Node B", "Node C"], random.Random(10))
chain = Blockchain([Validator(name, 100) for name in network.node_names])
collection = NFTCollection("class-photos.nft", creator="Alice", royalty_bps=500)

network.broadcast(
    Transaction("tx-deploy-nft", "Deploy NFTCollection at class-photos.nft"),
    origin="Node A",
)
_, deployment_block = network.include("Node A", "tx-deploy-nft", chain)

print(f"Off-chain file: {artifact_name} ({len(artifact_bytes)} bytes)")
print(f"SHA-256: {artifact_hash}")
print(f"Digest length: {len(artifact_hash)} hex characters")
print(f"Deployment receipt: Block #{deployment_block.index}")
print("The hash lets us check the file. It does not store the file.")

Off-chain file: class_graduation.txt (105 bytes)
SHA-256: 67b3b12033db0241f7db0cd2ed0c85a8e94038f33d1938954a5b31a51819a067
Digest length: 64 hex characters
Deployment receipt: Block #1
The hash lets us check the file. It does not store the file.


## Mint the first token

We create token #1 and assign it to Alice. Minting records the token's owner and metadata. It does not create the photo, verify who took it, or set a market price.

Our helper records the `mint` request in a block and then runs it. The Python collection holds the current ownership state. The block records the request and its metadata, including the link and hash. The file bytes stay outside the chain.

Token #1 needs a collection name to identify it. Token #1 in `class-photos.nft` is different from token #1 in another collection. On a real network, the chain, contract address, and token ID together identify the token. [ERC-721 identifiers](https://eips.ethereum.org/EIPS/eip-721).


In [5]:
token_id, _, mint_block = submit_call(
    network,
    chain,
    "Node A",
    "tx-mint-alice",
    collection,
    "mint",
    to="Alice",
    metadata=metadata,
    caller="Alice",
)

print(f"Minted ({collection.address}, #{token_id}) -> {collection.owner_of(token_id)}")
print(f"Title: {collection.metadata_of(token_id).name}")
print(f"Pointer: {collection.metadata_of(token_id).uri}")
print(f"Royalty quote rate: {collection.royalty_bps / 100:.1f}%")
print(f"Block #{mint_block.index} records: {mint_block.data}")

Minted (class-photos.nft, #1) -> Alice
Title: Class Graduation Photo
Pointer: https://gallery.invalid/class-graduation.txt
Royalty quote rate: 5.0%
Block #2 records: class-photos.nft.mint(to='Alice', metadata=TokenMetadata(name='Class Graduation Photo', description='The whole class together. Almost everyone has their eyes open.', content_hash='67b3b12033db0241f7db0cd2ed0c85a8e94038f33d1938954a5b31a51819a067', uri='https://gallery.invalid/class-graduation.txt'), caller='Alice')


## Copying the photo does not transfer the token

Jacob saves a copy of the file. His copy has the same bytes and the same hash. He really does have the same picture.

The token record still names Alice as its owner. When Jacob tries to transfer the token to himself, the contract checks the owner and rejects the request. Saving the photo never called the transfer method.

> **Before running:** What will `owner_of(1)` return after the copy? Will the rejected transfer change it?

In this model, names represent wallet identities. A real contract gets its caller from the transaction context. Passing `caller="Alice"` here makes the rule visible, but typing someone's name is not real authentication.

The block can record that Jacob attempted a call even though it failed. The attempt is in the log; the token's owner remains Alice.


In [6]:
jacob_copy = bytes(bytearray(artifact_bytes))
print(f"Identical bytes? {jacob_copy == artifact_bytes}")
print(
    f"Copy matches the token's hash? {collection.verify_content(token_id, jacob_copy)}"
)
print(f"Owner after right-click save: {collection.owner_of(token_id)}")

blocks_before_attempt = len(chain.chain)
try:
    submit_call(
        network,
        chain,
        "Node A",
        "tx-jacob-tries-theft",
        collection,
        "transfer",
        token_id=token_id,
        caller="Jacob",
        to="Jacob",
    )
except NotTheOwnerError as error:
    print(f"{type(error).__name__}: {error}")

print(f"Owner after rejected transfer: {collection.owner_of(token_id)}")
print(f"Attempt receipts added: {len(chain.chain) - blocks_before_attempt}")
print("The attempted call was recorded, but ownership did not change.")

Identical bytes? True
Copy matches the token's hash? True
Owner after right-click save: Alice
NotTheOwnerError: Jacob does not own token #1; Alice does.
Owner after rejected transfer: Alice
Attempt receipts added: 1
The attempted call was recorded, but ownership did not change.


## Sell the token and calculate a royalty

Alice sells token #1 to Bob for **2 ETH**. Bob later sells it to Wei Yan for **50 ETH**. These are fictional prices. The photo is unchanged between sales; the second buyer is simply willing to pay more.

A **creator royalty** is a share of a sale allocated to the original creator. Our collection quotes 5%:

- On the 2 ETH sale, the seller's share is 1.9 ETH and the creator's share is 0.1 ETH. Alice is both, so her combined allocation is 2 ETH.
- On Bob's 50 ETH resale, Bob's share is 47.5 ETH and Alice's creator share is 2.5 ETH.

If a marketplace pays that split, Alice can receive money from a later sale even after selling her token. We will also see why quoting a royalty does not guarantee payment.

### You can pay with a stablecoin too

An NFT can also be bought using a **stablecoin**: a fungible token designed to track a reference value, such as a fiat currency. A dollar-pegged stablecoin aims to track the US dollar, so a purchase can be priced in dollar-like units on-chain. The peg depends on the stablecoin's design and backing.

Suppose Bob buys token #1 for 100 stablecoin units. Before fees and royalties, two things need to happen:

- Bob sends 100 stablecoin units to Alice. Their currency balances change.
- Alice transfers NFT #1 to Bob. That token's owner changes.

The payment amount is fungible. The purchased token is individually identified. A marketplace needs to coordinate payment and transfer so both complete together. Otherwise someone could pay and still be waiting for the photo token, which is not an improvement on buying things normally.

The royalty can use the payment currency. At 5%, a 100-unit stablecoin sale would allocate 95 units to the seller and 5 to the creator. [ERC-2981](https://eips.ethereum.org/EIPS/eip-2981).

The code below uses ETH amounts. It changes ownership and calculates allocations, but has no currency balance ledger and makes no payment. Actual settlement would need another operation.


In [7]:
first_sale, _, _ = submit_call(
    network,
    chain,
    "Node A",
    "tx-sale-to-bob",
    collection,
    "sell",
    token_id=token_id,
    caller="Alice",
    to="Bob",
    price_eth=Decimal("2"),
)
resale, _, _ = submit_call(
    network,
    chain,
    "Node A",
    "tx-bob-flips",
    collection,
    "sell",
    token_id=token_id,
    caller="Bob",
    to="Wei Yan",
    price_eth=Decimal("50"),
)

for label, sale in [("Alice -> Bob", first_sale), ("Bob -> Wei Yan", resale)]:
    print(f"{label}: price {sale['price_eth']} ETH")
    print(f"  Seller allocation: {sale['seller_receives']:.3f} ETH")
    print(
        f"  Creator allocation: {sale['creator_royalty']:.3f} ETH -> {sale['creator']}"
    )

print(f"Current owner: {collection.owner_of(token_id)}")
print("Payout arithmetic completed. Actual ETH settlement: outside this toy.")

Alice -> Bob: price 2 ETH
  Seller allocation: 1.900 ETH
  Creator allocation: 0.100 ETH -> Alice
Bob -> Wei Yan: price 50 ETH
  Seller allocation: 47.500 ETH
  Creator allocation: 2.500 ETH -> Alice
Current owner: Wei Yan
Payout arithmetic completed. Actual ETH settlement: outside this toy.


## Why royalties can be skipped

ERC-2981 lets a marketplace ask how much royalty to pay and who should receive it. It does not force payment on every transfer. A transfer could be a gift, a move between someone's wallets, or part of a sale arranged elsewhere. [ERC-2981](https://eips.ethereum.org/EIPS/eip-2981).

Our `sell` method calculates a split. Our `transfer` method does not ask for a price or calculate a royalty. Wei Yan can use that method to move the token to Bob. Even if they had separately agreed on a payment, this transfer would not collect a creator's share.

A royalty quote is useful information. The marketplace still needs to make the payment.

> **Before running:** Does a direct transfer ask for a price? Does it pay Alice anything?


In [8]:
receiver, quoted_royalty = collection.royalty_info(token_id, Decimal("50"))
print(f"At a 50 ETH price, the quote would be {quoted_royalty:.3f} ETH to {receiver}.")

submit_call(
    network,
    chain,
    "Node A",
    "tx-direct-transfer",
    collection,
    "transfer",
    token_id=token_id,
    caller="Wei Yan",
    to="Bob",
)

print(f"Owner after direct transfer: {collection.owner_of(token_id)}")
print("This path asked for no price and collected no royalty.")
print("A quote is information. A payment is an additional operation.")

At a 50 ETH price, the quote would be 2.500 ETH to Alice.
Owner after direct transfer: Bob
This path asked for no price and collected no royalty.
A quote is information. A payment is an additional operation.


## Several NFTs can link to the same picture

A collection does not need a different picture for every token. Each token has its own ID and owner, even when two or more point to exactly the same file.

We demonstrate both cases below. Token #2 points to a second graduation photo. Token #3 points to the **same file as token #1**. After minting, our records will look like this:

| Token | Owner | Linked file |
| --- | --- | --- |
| #1 | Bob | Original graduation photo |
| #2 | Alice | A second graduation photo |
| #3 | Alice | Original graduation photo again |

Tokens #1 and #3 have separate ownership records. Transferring #1 does not transfer #3 or stop anyone copying their shared picture.

This is how a collection can contain many NFTs linked to one image. The same image does not make them the same token. To promise a limited edition, the contract would need a rule limiting how many can be minted. Our contract has no such limit.


In [9]:
second_bytes = b"Class graduation photo: a second shot, with fewer people blinking."
second_metadata = TokenMetadata(
    "Class Graduation Photo: Second Shot",
    "Same class, a second photo.",
    hashlib.sha256(second_bytes).hexdigest(),
    "https://gallery.invalid/class-graduation-second.txt",
)
off_chain_files[second_metadata.uri] = second_bytes
second_id, _, _ = submit_call(
    network,
    chain,
    "Node A",
    "tx-mint-second-photo",
    collection,
    "mint",
    to="Alice",
    metadata=second_metadata,
    caller="Alice",
)
third_id, _, _ = submit_call(
    network,
    chain,
    "Node A",
    "tx-mint-same-file-again",
    collection,
    "mint",
    to="Alice",
    metadata=metadata,
    caller="Alice",
)

for existing_id in [token_id, second_id, third_id]:
    print(
        f"Token #{existing_id}: owner={collection.owner_of(existing_id)}, "
        f"title={collection.metadata_of(existing_id).name!r}"
    )
print(
    "Tokens #1 and #3 point at identical bytes?",
    collection.metadata_of(token_id).content_hash
    == collection.metadata_of(third_id).content_hash,
)
try:
    collection.owner_of(999)
except TokenDoesNotExistError as error:
    print(f"{type(error).__name__}: {error}")

Token #1: owner=Bob, title='Class Graduation Photo'
Token #2: owner=Alice, title='Class Graduation Photo: Second Shot'
Token #3: owner=Alice, title='Class Graduation Photo'
Tokens #1 and #3 point at identical bytes? True
TokenDoesNotExistError: Token #999 does not exist.


## Someone else can mint a token linked to your photo

Jacob creates another collection and mints its token #1 using the same file. He owns that token, but it is a different token from #1 in Alice's collection. The collection addresses and ownership records differ.

Jacob's file passes the hash check because its bytes match. That does not prove he took the photo or has permission to use it. It only checks the bytes against the recorded hash.

Applications need to check which collection they recognise. For example, a membership check should look for a token in the accepted collection, not any token linked to a matching image. Otherwise copying the club's logo would be a surprisingly effective application form.


In [10]:
imitation = NFTCollection("class-photos-copy.nft", creator="Jacob")
fake_id, _, _ = submit_call(
    network,
    chain,
    "Node A",
    "tx-imitation-mint",
    imitation,
    "mint",
    to="Jacob",
    metadata=metadata,
    caller="Jacob",
)

print(
    f"Recognised entry: ({collection.address}, #{token_id}) -> {collection.owner_of(token_id)}"
)
print(
    f"Imitation entry: ({imitation.address}, #{fake_id}) -> {imitation.owner_of(fake_id)}"
)
print(f"Imitation file hash matches? {imitation.verify_content(fake_id, jacob_copy)}")
print(f"Jacob owns the recognised token? {collection.owner_of(token_id) == 'Jacob'}")
print("The file bytes match. This does not prove who took the photo.")

Recognised entry: (class-photos.nft, #1) -> Bob
Imitation entry: (class-photos-copy.nft, #1) -> Jacob
Imitation file hash matches? True
Jacob owns the recognised token? False
The file bytes match. This does not prove who took the photo.


## Check the file, then remove it from the host

First we edit the file. Its hash no longer matches the stored hash, so our check returns `False`.

Then we delete the original file from our hosting dictionary. The token still has its owner, hash, and link. The chain still passes its checks. But the host cannot supply the file at that address.

> **Before running:** Does deleting the off-chain file add a block or change an existing block's hash?


In [11]:
edited_bytes = artifact_bytes.replace(b"Students", b"Cardboard cutouts")
print(f"Original matches? {collection.verify_content(token_id, artifact_bytes)}")
print(f"Edited version matches? {collection.verify_content(token_id, edited_bytes)}")

blocks_before_host_failure = len(chain.chain)
off_chain_files.pop(metadata.uri)

print(f"File available from our host? {metadata.uri in off_chain_files}")
print(f"Owner still recorded: {collection.owner_of(token_id)}")
print(f"URI still recorded: {collection.metadata_of(token_id).uri}")
print(f"Blocks added by host failure: {len(chain.chain) - blocks_before_host_failure}")
print(f"Chain check: {chain.is_valid()}")
print(
    f"Jacob's saved copy still verifies? {collection.verify_content(token_id, jacob_copy)}"
)
print("The host lost its file, but the saved copy is still usable.")

Original matches? True
Edited version matches? False
File available from our host? False
Owner still recorded: Bob
URI still recorded: https://gallery.invalid/class-graduation.txt
Blocks added by host failure: 0
Chain check: (True, 'Chain is valid.')
Jacob's saved copy still verifies? True
The host lost its file, but the saved copy is still usable.


Jacob's saved copy still matches. Deleting one host's file has not deleted every copy. It has made the file unavailable from that host.

A hash can check bytes you have. It cannot recover missing bytes. Ownership, file integrity, and file availability need separate checks.

**IPFS** is a distributed file system often used for NFT content. It identifies data by its content, but still needs providers to keep and serve the files. **Pinning** tells a node to retain a file rather than discard it during cleanup. A content identifier alone does not keep the file online. [IPFS persistence and pinning](https://docs.ipfs.tech/concepts/persistence/).

Someone still has to store the graduation photo. Putting its link on a blockchain does not volunteer anyone to pay the hosting bill.


## A high sale price does not describe every NFT

People may buy NFTs to collect them, use them for access, or resell them. The contract records the owner. It does not guarantee that another buyer will pay the same price later.

One expensive sale also does not tell us the typical value of all NFTs. An outlier can raise an average while most items have much lower values.

Here is a **made-up example**, not market data: 990 items have zero value, nine have modest values, and one is worth a million dollars. It shows how an outlier affects the average. It does not estimate NFT prices, and an unsold listing in a real market is not automatically a completed sale at zero.

> **Before running:** Could the average exceed $1,000 while the middle value, or median, is $0?


In [12]:
toy_values_usd = [0] * 990 + [100] * 9 + [1_000_000]
print("ILLUSTRATIVE VALUES ONLY -- no live market data")
print(f"Items: {len(toy_values_usd):,}")
print(f"Mean: ${statistics.mean(toy_values_usd):,.2f}")
print(f"Median: ${statistics.median(toy_values_usd):,.2f}")
print(f"Largest: ${max(toy_values_usd):,.0f}")
print(
    "One item is valued at a million dollars; most in this example are valued at zero."
)
print("The largest value raises the average without changing the median.")

ILLUSTRATIVE VALUES ONLY -- no live market data
Items: 1,000
Mean: $1,000.90
Median: $0.00
Largest: $1,000,000
One item is valued at a million dollars; most in this example are valued at zero.
The largest value raises the average without changing the median.


## Uses beyond pictures

An NFT can identify a particular ticket, membership, game item, or other record. An application can use its owner field to decide who may access or use that item.

| Use | What the record helps track | What else is needed |
| --- | --- | --- |
| Ticket | Who holds this admission token | An accepted issuer, admission checks, and rules preventing reuse |
| Membership | Who holds a qualifying token | Access checks and membership benefits |
| Game item | Who holds this particular item | A game that recognises the item |
| Credential | A record from a particular issuer | Trust in the issuer, privacy, and appropriate transfer or revocation rules |

Owning our photo token means being named as its owner in the collection. We have included no copyright assignment, exclusive image-use rights, or licence terms. Those would require a separate arrangement.

For each use, ask what the token represents and what the application actually provides. Owning a membership token is less exciting if the only benefit is being told that benefits are coming soon.


## What this example implements

This is a Python teaching model, not a deployed Ethereum contract. Names stand in for wallet identities, and Python objects store the current state. Blocks record requested calls, including an attempt that might fail.

Only the creator can mint in our collection. The public methods leave existing metadata unchanged, allow repeated file hashes, and impose no edition cap. We omit approvals, safe-transfer callbacks, burns, transaction rollback, fees, and actual payments. A dictionary represents the external file host.

The examples show four things separately: who owns a token, which file it refers to, whether the bytes match, and whether the file is available.


## Check your understanding

1. Jacob has the exact file bytes. Why can he not transfer Alice's token?
2. Why do we need a collection and chain as well as the token ID?
3. Can a collection contain many NFTs linked to the same photo?
4. Is a limit on how many tokens can be minted the same as non-fungibility?
5. What did our sales change, and what did they merely calculate?
6. How can a stablecoin be used to buy an NFT?
7. Why does a direct transfer skip our royalty calculation?
8. Why does the chain stay valid when the host loses the file?
9. What does a matching file hash prove?

<details>
<summary>Show the answers</summary>

1. The transfer method checks the token's current owner. Having the file does not change that owner.
2. Different collections and chains can each have a token with the same ID.
3. Yes. Each still has its own ID and ownership record. Our #1 and #3 link to the same file.
4. No. A minting limit controls supply. Non-fungibility concerns individual identity and interchangeability.
5. They changed ownership and calculated proposed payouts. No ETH balances moved.
6. The buyer pays a fungible stablecoin amount and receives ownership of the individually identified NFT. A marketplace needs to coordinate both operations.
7. It asks for no sale price and calls no royalty or payment operation.
8. Off-chain deletion changes neither the block data nor the hash links.
9. It proves the supplied bytes match the stored fingerprint. It does not establish authorship, value, permission to use the image, or future availability.

</details>

An NFT records ownership of a particular token. That token may share its picture with other NFTs, and anyone may still copy the file. A useful application needs clear rules for what owning the token lets its holder do.


## Sources and further reading

- [ERC-721: Non-Fungible Token Standard](https://eips.ethereum.org/EIPS/eip-721) — individual ownership, transfers, token identifiers, and optional metadata.
- [ERC-2981: NFT Royalty Standard](https://eips.ethereum.org/EIPS/eip-2981) — royalty payment information, with payment handled separately.
- [IPFS: Persistence, permanence, and pinning](https://docs.ipfs.tech/concepts/persistence/) — how content is kept available.

All NFT-specific classes are defined inline in this notebook. Their reusable copies are in [`blockchain_lib/nft.py`](../blockchain_lib/nft.py).
